# ML sobre las features: versión LIGERA

Mismo flujo que el notebook anterior (árbol de decisión → selección top-K → baseline y otros modelos → ablación →
evaluación en val + diagnósticos), pero con menos coste:

| Qué pesaba | Qué se hace aquí                                                                                                                |
|---|---------------------------------------------------------------------------------------------------------------------------------|
| La selección de features se repetía en **cada modelo y cada fold** (20 árboles × modelos × folds) | Se calcula **una vez por fold** y la comparten todos los modelos                                                                |
| CV con las 15 195 filas | CV y ablación sobre una **submuestra estratificada** (`CV_ROWS`); el ajuste **final** usa all train y la evaluación usa all val |
| SVM-RBF (crece con el cuadrado de las filas) | Se quita                                                                                                                        |
| Random Forest de 300 árboles, ablación con Random Forest | 150 árboles; ablación con HistGradientBoosting                                                                                  |
| Importancia por permutación sobre las ~300 features | Solo sobre las K seleccionadas                                                                                                  |
| 5 folds | 3 folds, ejecutados en paralelo                                                                                                 |

La selección sigue haciéndose **dentro de cada fold** (con los datos de entrenamiento de ese fold), así que las métricas
de CV no salen infladas. Cada etapa imprime lo que ha tardado.

Etiqueta: `label` (nevus = 0, others = 1). Se excluyen `name`, `class`, `subtype` (delata la clase) y `qc_*`.

In [1]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import QuantileTransformer
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore")

# ---------------- Configuración ----------------
ROOT = Path(r"C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG")
TRAIN_CSV = ROOT / "features_completes2" / "features_train.csv"
VAL_CSV = ROOT / "features_completes2" / "features_val.csv"
MANIFEST = ROOT / "preprocessed2" / "manifest.csv"        # solo para las celdas de diagnóstico del final
OUT_DIR = ROOT / "ml" / "ml_results_completes2_fast"

K_FEATURES = 30          # features que se quedan tras la selección
CV_FOLDS = 3
CV_ROWS = 6000           # filas de train para CV y ablación (None = todas). El ajuste FINAL usa all train
SEL_ROWS = 3000          # filas con las que se puntúan las features en cada fold
SEL_ROUNDS = 10          # árboles bootstrap por fold para puntuar features
SEED = 42
N_JOBS = 4
OUT_DIR.mkdir(parents=True, exist_ok=True)

_t0 = time.time()
def tic(msg=""):
    print(f"[{time.time() - _t0:6.0f}s] {msg}")

## 1. Carga de datos

In [2]:
tr, va = pd.read_csv(TRAIN_CSV), pd.read_csv(VAL_CSV)

PREFIXES = ("shape_", "color_", "tex_", "gcolor_", "gtex_")
FEATS = [c for c in tr.columns if c.startswith(PREFIXES)]
assert FEATS == [c for c in va.columns if c.startswith(PREFIXES)], "train y val tienen columnas distintas"
family = lambda c: c.split("_")[0]

Xtr = tr[FEATS].replace([np.inf, -np.inf], np.nan).astype(np.float32)
Xva = va[FEATS].replace([np.inf, -np.inf], np.nan).astype(np.float32)
ytr, yva = tr["label"].values, va["label"].values
Xtr_np, Xva_np = Xtr.values, Xva.values

# submuestra estratificada para la CV y la ablación
if CV_ROWS and CV_ROWS < len(tr):
    cv_idx, _ = train_test_split(np.arange(len(tr)), train_size=CV_ROWS, stratify=ytr, random_state=SEED)
else:
    cv_idx = np.arange(len(tr))
Xcv_np, ycv = Xtr_np[cv_idx], ytr[cv_idx]

print(f"train: {Xtr.shape} (others = {ytr.mean():.1%})   val: {Xva.shape} (others = {yva.mean():.1%})")
print(f"CV/ablación sobre {len(cv_idx)} filas de train; ajuste final con {len(tr)}; evaluación con {len(va)} de val")
print("features por familia:", pd.Series([family(c) for c in FEATS]).value_counts().to_dict())
print("NaN en features: train", int(Xtr.isna().sum().sum()), "| val", int(Xva.isna().sum().sum()))

skf = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
FAM_COLOR = {"shape": "#1f77b4", "color": "#d62728", "tex": "#2ca02c", "gcolor": "#ff9896", "gtex": "#98df8a"}

train: (15195, 420) (others = 49.2%)   val: (3796, 420) (others = 49.1%)
CV/ablación sobre 6000 filas de train; ajuste final con 15195; evaluación con 3796 de val
features por familia: {'tex': 95, 'gtex': 95, 'color': 83, 'shape': 76, 'gcolor': 71}
NaN en features: train 0 | val 0


## 2. Árbol de decisión: ¿qué features son más útiles?
Un árbol poco profundo (`max_depth=5`). Un solo árbol es inestable, así que también se mira si el ranking se repite entre folds.

In [ ]:
imp_full = SimpleImputer(strategy="median").fit(Xtr_np)
Xtr_i = imp_full.transform(Xtr_np)

tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED).fit(Xtr_i, ytr)
imp_tree = pd.Series(tree.feature_importances_, index=FEATS).sort_values(ascending=False)

fig, axs = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [3, 1]})
top = imp_tree.head(25)[::-1]
axs[0].barh(top.index, top.values, color=[FAM_COLOR[family(c)] for c in top.index])
axs[0].set_title("Árbol de decisión (max_depth=5): top 25 features")
fam_imp = imp_tree.groupby(lambda c: family(c)).sum()
axs[1].bar(fam_imp.index, fam_imp.values, color=[FAM_COLOR[f] for f in fam_imp.index])
axs[1].set_title("Importancia total por familia")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(tree, max_depth=3, feature_names=FEATS, class_names=["nevus", "others"], filled=True, fontsize=8, ax=ax)
ax.set_title("Primeros 3 niveles del árbol")
plt.show()
tic("árbol de decisión")

### Estabilidad del ranking entre folds
Un árbol por fold (con sus datos de entrenamiento). Una feature útil de verdad debería salir arriba en casi todos.

In [ ]:
Xcv_i = SimpleImputer(strategy="median").fit_transform(Xcv_np)
rows, aucs = [], []
for f, (a, b) in enumerate(skf.split(Xcv_i, ycv)):
    t = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED).fit(Xcv_i[a], ycv[a])
    rows.append(pd.Series(t.feature_importances_, index=FEATS, name=f))
    aucs.append(roc_auc_score(ycv[b], t.predict_proba(Xcv_i[b])[:, 1]))
imp_cv = pd.DataFrame(rows)
stab = pd.DataFrame({"importancia_media": imp_cv.mean(), "importancia_std": imp_cv.std(),
                     f"folds_en_top15 (de {CV_FOLDS})": (imp_cv.rank(axis=1, ascending=False) <= 15).sum()})
print(f"AUC del árbol solo (CV {CV_FOLDS} folds): {np.mean(aucs):.3f} ± {np.std(aucs):.3f}\n")
print(stab.sort_values("importancia_media", ascending=False).head(20).round(3).to_string())

## 3. Selección de las top-K features
`rank_features` promedia la importancia de varios árboles entrenados con muestras bootstrap (más estable que un árbol suelto).
En la CV se calcula **una vez por fold** con los datos de entrenamiento de ese fold y la reutilizan todos los modelos.

In [ ]:
def rank_features(X, y, rounds, rows=None, max_depth=6, seed=0):
    """Importancia media de `rounds` árboles de decisión entrenados con bootstrap (sobre `rows` filas)."""
    rng = np.random.RandomState(seed)
    n = len(y)
    imp = np.zeros(X.shape[1])
    for _ in range(rounds):
        idx = rng.choice(n, min(rows or n, n), replace=True)
        t = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=10,
                                   random_state=rng.randint(1_000_000_000)).fit(X[idx], y[idx])
        imp += t.feature_importances_
    return imp / rounds


# ranking final con all train (es el que usa el modelo final; la CV lo recalcula dentro de cada fold)
rank_full = rank_features(Xtr_i, ytr, rounds=20, rows=6000, seed=SEED)
sel_imp = pd.Series(rank_full, index=FEATS).sort_values(ascending=False)
top_f = np.argsort(-rank_full)[:K_FEATURES]
SELECTED = [FEATS[i] for i in top_f]
sel_imp.head(K_FEATURES).rename("importancia").to_csv(OUT_DIR / "selected_features.csv")

top = sel_imp.head(K_FEATURES)[::-1]
plt.figure(figsize=(9, 7))
plt.barh(top.index, top.values, color=[FAM_COLOR[family(c)] for c in top.index])
plt.title(f"Top {K_FEATURES} features (promedio de árboles bootstrap)")
plt.tight_layout(); plt.show()
print("Familias en el top:", pd.Series([family(c) for c in SELECTED]).value_counts().to_dict())
tic("selección de features")

## 4. Baseline y otros modelos (CV)
- Por fold: imputación (mediana) → ranking de features (una vez) → transformación cuantil → todos los modelos.
- **Baseline**: regresión logística. `Dummy` es el suelo (AUC 0.5).
- Con **todas** las features solo se prueban los modelos rápidos; el KNN solo con las top-K.
- Métricas: AUC, balanced accuracy, F1 y **recall de `others`** (clase positiva = 1).

In [ ]:
MODELS = {   # nombre: (fábrica(n_jobs), ¿necesita escalado?, ¿también con todas las features?)
    "Dummy": (lambda nj: DummyClassifier(strategy="prior"), False, True),
    "LogReg (baseline)": (lambda nj: LogisticRegression(C=0.5, max_iter=2000), True, True),
    "KNN": (lambda nj: KNeighborsClassifier(n_neighbors=15), True, False),
    "Decision Tree": (lambda nj: DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED), False, True),
    "Random Forest": (lambda nj: RandomForestClassifier(n_estimators=150, min_samples_leaf=3, max_features="sqrt",
                                                        n_jobs=nj, random_state=SEED), False, True),
    "HistGradientBoosting": (lambda nj: HistGradientBoostingClassifier(learning_rate=0.1, max_iter=150, max_depth=4,
                                                                      early_stopping=False, random_state=SEED), False, True),
}


def get_score(m, X):
    return m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)


def metrics(y, s, pred):
    return {"auc": roc_auc_score(y, s), "bal_acc": balanced_accuracy_score(y, pred),
            "f1": f1_score(y, pred), "recall_others": recall_score(y, pred)}


def new_qt():
    return QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED)


def fold_job(Xa, ya, Xb, yb, nj):
    """Un fold completo: preprocesado y selección UNA vez, y todos los modelos con ese preprocesado."""
    imp = SimpleImputer(strategy="median").fit(Xa)
    A, B = imp.transform(Xa), imp.transform(Xb)
    top = np.argsort(-rank_features(A, ya, SEL_ROUNDS, SEL_ROWS, SEED))[:K_FEATURES]
    qa, qt = new_qt().fit(A), new_qt().fit(A[:, top])
    mats = {"all": (A, B, qa.transform(A), qa.transform(B)),
            "top": (A[:, top], B[:, top], qt.transform(A[:, top]), qt.transform(B[:, top]))}
    out = []
    for setting, (Ra, Rb, Sa, Sb) in mats.items():
        for name, (make, scale, also_all) in MODELS.items():
            if setting == "all" and not also_all:
                continue
            m = make(nj).fit(Sa if scale else Ra, ya)
            Xe = Sb if scale else Rb
            out.append({"setting": setting, "model": name, **metrics(yb, get_score(m, Xe), m.predict(Xe))})
    return out


res = Parallel(n_jobs=min(N_JOBS, CV_FOLDS))(
    delayed(fold_job)(Xcv_np[a], ycv[a], Xcv_np[b], ycv[b], 1) for a, b in skf.split(Xcv_np, ycv))
cv = pd.DataFrame([r for fold in res for r in fold])


def cv_table(setting):
    g = cv[cv["setting"] == setting].groupby("model")
    mean, std = g.mean(numeric_only=True), g.std(numeric_only=True)
    t = pd.DataFrame({m: [f"{mean.loc[k, m]:.3f} ± {std.loc[k, m]:.3f}" for k in mean.index] for m in mean.columns},
                     index=mean.index)
    t["_auc"] = mean["auc"]
    return t.sort_values("_auc", ascending=False)


res_all, res_top = cv_table("all"), cv_table("top")
print(f"CV {CV_FOLDS} folds sobre {len(cv_idx)} filas\n")
print(f"--- TODAS las features ({len(FEATS)}) ---")
print(res_all.drop(columns="_auc").to_string(), "\n")
print(f"--- TOP-{K_FEATURES} (selección dentro de cada fold) ---")
print(res_top.drop(columns="_auc").to_string())
res_all.drop(columns="_auc").to_csv(OUT_DIR / "cv_all_features.csv")
res_top.drop(columns="_auc").to_csv(OUT_DIR / f"cv_top{K_FEATURES}.csv")
tic("validación cruzada")

## 5. Ablación por familias
HistGradientBoosting (rápido y tolera NaN) con cada familia sola y combinadas.
Ojo con `gcolor`/`gtex` (imagen entera): pueden captar la cámara, la iluminación o el fondo y no la lesión.

In [ ]:
fam_cols = {f: [c for c in FEATS if family(c) == f] for f in ["shape", "color", "tex", "gcolor", "gtex"]}
lesion = fam_cols["shape"] + fam_cols["color"] + fam_cols["tex"]
imagen = fam_cols["gcolor"] + fam_cols["gtex"]
subsets = {"solo forma": fam_cols["shape"], "solo color": fam_cols["color"], "solo textura": fam_cols["tex"],
           "solo gcolor": fam_cols["gcolor"], "solo gtex": fam_cols["gtex"],
           "forma + color": fam_cols["shape"] + fam_cols["color"], "color + textura": fam_cols["color"] + fam_cols["tex"],
           "lesión (shape+color+tex)": lesion, "imagen entera (gcolor+gtex)": imagen, "todas": FEATS}
pos = {c: i for i, c in enumerate(FEATS)}
abl = {}
for name, cols in subsets.items():
    if not cols:
        continue
    r = cross_validate(MODELS["HistGradientBoosting"][0](1), Xcv_np[:, [pos[c] for c in cols]], ycv, cv=skf,
                       scoring={"auc": "roc_auc", "bal_acc": "balanced_accuracy", "recall_others": "recall"}, n_jobs=N_JOBS)
    abl[name] = {"n_features": len(cols), **{m: f"{r['test_' + m].mean():.3f} ± {r['test_' + m].std():.3f}"
                                             for m in ("auc", "bal_acc", "recall_others")}}
abl = pd.DataFrame(abl).T
print(abl.to_string())
abl.to_csv(OUT_DIR / "ablation_families.csv")
tic("ablación")

## 6. Ajuste final (all train) y evaluación en val
Preprocesado y selección con **all train**; cada modelo se entrena una vez y se evalúa una sola vez en all val.
El IC del AUC se obtiene por bootstrap sobre las imágenes de val.

In [ ]:
A, B = Xtr_i, imp_full.transform(Xva_np)
Ra, Rb = A[:, top_f], B[:, top_f]
qt_f = new_qt().fit(Ra)
Sa, Sb = qt_f.transform(Ra), qt_f.transform(Rb)


def boot_ci(y, s, n=1000, seed=0):
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n):
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i])) == 2:
            vals.append(roc_auc_score(y[i], s[i]))
    return np.percentile(vals, [2.5, 97.5])


fitted, scores, VAL_X, rows = {}, {}, {}, []
for name, (make, scale, _) in MODELS.items():
    m = make(N_JOBS).fit(Sa if scale else Ra, ytr)
    Xe = Sb if scale else Rb
    s, pred = get_score(m, Xe), m.predict(Xe)
    lo, hi = boot_ci(yva, s)
    fitted[name], scores[name], VAL_X[name] = m, s, Xe
    rows.append({"modelo": name, "AUC": roc_auc_score(yva, s), "AUC IC95%": f"[{lo:.3f}, {hi:.3f}]",
                 "bal_acc": balanced_accuracy_score(yva, pred), "F1": f1_score(yva, pred),
                 "recall_others": recall_score(yva, pred), "precision_others": precision_score(yva, pred, zero_division=0)})
val_res = pd.DataFrame(rows).set_index("modelo").sort_values("AUC", ascending=False)
print(f"Val ({len(Xva)} imágenes), modelos con top-{K_FEATURES}:\n")
print(val_res.round(3).to_string())
val_res.round(4).to_csv(OUT_DIR / f"val_top{K_FEATURES}.csv")
tic("ajuste final y val")

In [ ]:
best = res_top.drop(index="Dummy", errors="ignore").index[0]       # mejor según AUC en CV (no según val)
print("Mejor modelo en CV:", best)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(yva, s)
    axs[0].plot(fpr, tpr, lw=2 if name == best else 1, label=f"{name} ({roc_auc_score(yva, s):.3f})")
axs[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axs[0].set(xlabel="Falsos positivos", ylabel="Recall (others)", title="ROC en val"); axs[0].legend(fontsize=8)
ConfusionMatrixDisplay.from_predictions(yva, fitted[best].predict(VAL_X[best]), display_labels=["nevus", "others"],
                                        ax=axs[1], colorbar=False)
axs[1].set_title(f"Matriz de confusión: {best}")
plt.tight_layout(); plt.show()

### ¿Qué features usa realmente el mejor modelo?
Importancia por permutación en val, **solo sobre las K seleccionadas** (las demás no entran al modelo). Sirve también para
vigilar que no dominen `gcolor_`/`gtex_`, que podrían delatar la cámara y no la lesión.

In [ ]:
Xe = VAL_X[best]
rng = np.random.RandomState(SEED)
base = roc_auc_score(yva, get_score(fitted[best], Xe))
drops = {}
for j, c in enumerate(SELECTED):
    d = []
    for _ in range(3):
        Xp = Xe.copy()
        Xp[:, j] = rng.permutation(Xp[:, j])
        d.append(base - roc_auc_score(yva, get_score(fitted[best], Xp)))
    drops[c] = np.mean(d)
pi = pd.Series(drops).sort_values(ascending=False).head(15)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(pi.index, pi.values, color=[FAM_COLOR[family(c)] for c in pi.index])
plt.title(f"Importancia por permutación en val ({best})")
plt.xlabel("caída del AUC"); plt.tight_layout(); plt.show()
tic("importancia por permutación")

## 7. Diagnósticos (sesgo de fuente y rendimiento por subtipo)

In [ ]:
# 7a. ¿Predice la clase solo el tamaño original de la imagen? (cerca de 0.5 = bien)
man = pd.read_csv(MANIFEST).set_index("name")
cols = ["orig_h", "orig_w"]
m_tr, m_va = man.reindex(tr["name"])[cols], man.reindex(va["name"])[cols]
g = HistGradientBoostingClassifier(random_state=SEED).fit(m_tr, ytr)
print("AUC solo con el tamaño original:", roc_auc_score(yva, g.predict_proba(m_va)[:, 1]))

In [ ]:
# 7b. Acierto por subtipo (nev = especificidad; el resto = recall)
pred = fitted[best].predict(VAL_X[best])
tmp = va[["subtype"]].assign(acierto=(pred == yva))
print(tmp.groupby("subtype")["acierto"].agg(["mean", "size"]).round(3))

In [ ]:
# 7c. AUC dentro de cada resolución original (si cae mucho, el modelo usa rasgos ligados a la fuente)
sz = man.reindex(va["name"])[["orig_h", "orig_w"]].reset_index(drop=True)
va_sz = va[["name", "class"]].reset_index(drop=True).join(sz)
va_sz["res"] = va_sz["orig_w"].astype(str) + "x" + va_sz["orig_h"].astype(str)
print(pd.crosstab(va_sz["res"], va_sz["class"]).sort_values("nevus", ascending=False).head(10))

s, rows = scores[best], []
for res_, g_ in va_sz.groupby("res"):
    y = yva[g_.index]
    if len(g_) >= 30 and 0 < y.mean() < 1:
        rows.append((res_, len(g_), round(y.mean(), 2), roc_auc_score(y, s[g_.index])))
print(pd.DataFrame(rows, columns=["res", "n", "frac_others", "AUC"]).round(3))

In [ ]:
# 7d. AUC melanoma vs nevus y sensibilidad a especificidad fija (umbral elegido mirando val: ligeramente optimista)
mm = va["subtype"].isin(["mel", "nev"]).values
print("AUC mel vs nev:", roc_auc_score(yva[mm], scores[best][mm]))

fpr, tpr, _ = roc_curve(yva, scores[best])
for sp in [0.80, 0.90, 0.95]:
    print(f"especificidad {sp:.0%}: sensibilidad {tpr[np.searchsorted(fpr, 1 - sp, side='right') - 1]:.3f}")
tic("fin")